# Mesh layers

Port of `packages/niivue/examples/mesh.layers.html`. Loads a cortical mesh with curvature and statistical overlay layers, then exposes threshold controls.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"
MESHES = f"{BASE_URL}/meshes"

nv = NiiVue(background_color=[0.2, 0.2, 0.3, 1.0], slice_type=4, backend="webgl2")

slice_type = widgets.Dropdown(
    options=[("Axial", 0), ("Coronal", 1), ("Sagittal", 2), ("A+C+S+R", 3), ("Render", 4)],
    value=4,
    description="Slice",
)
positive = widgets.FloatRangeSlider(
    value=[1.5, 5.0], min=0.0, max=10.0, step=0.1, description="Positive"
)
negative = widgets.FloatRangeSlider(
    value=[-2.0, -1.5], min=-10.0, max=0.0, step=0.1, description="Negative"
)
alpha_mode = widgets.Dropdown(
    options=[("Min-to-max", 0), ("Transparent below min", 1), ("Translucent below min", 2)],
    value=0,
    description="Alpha",
)


def update_slice(change):
    nv.slice_type = change["new"]


def update_thresholds(_change=None):
    pos_min, pos_max = positive.value
    neg_max, neg_min = negative.value
    nv.set_mesh_layer_property(
        0,
        1,
        {
            "calMin": pos_min,
            "calMax": pos_max,
            "cal_minNeg": neg_min,
            "cal_maxNeg": neg_max,
            "colormapType": alpha_mode.value,
        },
    )


slice_type.observe(update_slice, names="value")
positive.observe(update_thresholds, names="value")
negative.observe(update_thresholds, names="value")
alpha_mode.observe(update_thresholds, names="value")

controls = widgets.VBox([widgets.HBox([slice_type, alpha_mode]), positive, negative])
display(controls)
display(nv)

nv.load_meshes(
    [
        {
            "url": f"{MESHES}/BrainMesh_ICBM152.lh.mz3",
            "layers": [
                {
                    "url": f"{MESHES}/BrainMesh_ICBM152.lh.curv",
                    "colormap": "gray",
                    "calMin": 0.3,
                    "calMax": 0.5,
                    "opacity": 1,
                    "isColorbarVisible": False,
                },
                {
                    "url": f"{MESHES}/BrainMesh_ICBM152.lh.motor.mz3",
                    "calMin": 1.5,
                    "calMax": 5,
                    "cal_minNeg": -1.5,
                    "cal_maxNeg": -2,
                    "colormap": "warm",
                    "colormapNegative": "winter",
                    "opacity": 0.7,
                },
            ],
        }
    ]
)
update_thresholds()